# Platform setup

Create the catalog schemas and baseline Delta tables for the market macro lakehouse.

This notebook is create-only. To rebuild from scratch, drop the existing tables first and then rerun this notebook.

In [ ]:
dbutils.widgets.text("catalog", "market_macro")

catalog = dbutils.widgets.get("catalog").strip() or "market_macro"

schemas = [
    "brz_market",
    "slv_market",
    "gld_market",
    "brz_macro",
    "slv_macro",
    "gld_macro",
    "gld_cross",
    "obs",
]

def constraint_exists(table_name: str, constraint_name: str) -> bool:
    property_key = f"delta.constraints.{constraint_name}"
    properties = spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
    return any(row.key == property_key for row in properties)

table_specs = {
    "brz_market.raw_coinbase_ohlc_1d": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_market.raw_coinbase_ohlc_1d (
            product_id STRING NOT NULL COMMENT 'Trading pair like BTC-USD',
            bar_date DATE NOT NULL COMMENT 'UTC date of the OHLC bar',
            open DOUBLE,
            high DOUBLE,
            low DOUBLE,
            close DOUBLE,
            volume DOUBLE,
            source_window_start TIMESTAMP COMMENT 'Requested window start used by ingestion (UTC)',
            source_window_end TIMESTAMP COMMENT 'Requested window end used by ingestion (UTC)',
            ingested_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp (UTC)',
            run_id STRING NOT NULL COMMENT 'Pipeline run identifier',
            payload_hash STRING COMMENT 'Hash of raw API payload for audit/dedup'
        )
        USING DELTA
        COMMENT 'Bronze: raw daily OHLC bars ingested from Coinbase. Logical PK = (product_id, bar_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_coinbase_ohlc_keys": """
            ALTER TABLE `{catalog}`.brz_market.raw_coinbase_ohlc_1d
            ADD CONSTRAINT ck_coinbase_ohlc_keys
            CHECK (product_id IS NOT NULL AND bar_date IS NOT NULL)
            """,
            "ck_coinbase_ohlc_prices": """
            ALTER TABLE `{catalog}`.brz_market.raw_coinbase_ohlc_1d
            ADD CONSTRAINT ck_coinbase_ohlc_prices
            CHECK (
                (open IS NULL OR open >= 0) AND
                (high IS NULL OR high >= 0) AND
                (low IS NULL OR low >= 0) AND
                (close IS NULL OR close >= 0) AND
                (volume IS NULL OR volume >= 0)
            )
            """,
        },
    },
    "slv_market.crypto_ohlc_1d": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_market.crypto_ohlc_1d (
            product_id STRING NOT NULL COMMENT 'Logical PK part 1',
            bar_date DATE NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING NOT NULL COMMENT 'Parsed from product_id',
            quote_currency STRING NOT NULL COMMENT 'Parsed from product_id',
            open DOUBLE,
            high DOUBLE,
            low DOUBLE,
            close DOUBLE,
            volume DOUBLE,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Silver: cleaned daily crypto OHLC bars. Logical PK = (product_id, bar_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_slv_crypto_ohlc_keys": """
            ALTER TABLE `{catalog}`.slv_market.crypto_ohlc_1d
            ADD CONSTRAINT ck_slv_crypto_ohlc_keys
            CHECK (
                product_id IS NOT NULL AND bar_date IS NOT NULL AND
                base_asset IS NOT NULL AND quote_currency IS NOT NULL
            )
            """,
            "ck_slv_crypto_ohlc_prices": """
            ALTER TABLE `{catalog}`.slv_market.crypto_ohlc_1d
            ADD CONSTRAINT ck_slv_crypto_ohlc_prices
            CHECK (
                (open IS NULL OR open >= 0) AND
                (high IS NULL OR high >= 0) AND
                (low IS NULL OR low >= 0) AND
                (close IS NULL OR close >= 0) AND
                (volume IS NULL OR volume >= 0)
            )
            """,
        },
    },
    "slv_market.crypto_ohlc_1d_quarantine": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_market.crypto_ohlc_1d_quarantine (
            run_id STRING NOT NULL COMMENT 'Silver pipeline run identifier',
            product_id STRING NOT NULL COMMENT 'Source business key part 1',
            bar_date DATE NOT NULL COMMENT 'Source business key part 2',
            base_asset STRING COMMENT 'Parsed from product_id when available',
            quote_currency STRING COMMENT 'Parsed from product_id when available',
            open DOUBLE,
            high DOUBLE,
            low DOUBLE,
            close DOUBLE,
            volume DOUBLE,
            dq_reason STRING NOT NULL COMMENT 'Reason the row was rejected by Silver DQ rules',
            source_window_start TIMESTAMP COMMENT 'Bronze request window start (UTC)',
            source_window_end TIMESTAMP COMMENT 'Bronze request window end (UTC)',
            source_ingested_at TIMESTAMP NOT NULL COMMENT 'Bronze ingestion timestamp (UTC)',
            source_run_id STRING NOT NULL COMMENT 'Bronze pipeline run identifier',
            payload_hash STRING COMMENT 'Bronze payload hash for audit lineage',
            quarantined_at TIMESTAMP NOT NULL COMMENT 'Timestamp when the row entered quarantine'
        )
        USING DELTA
        COMMENT 'Silver quarantine for rejected crypto OHLC rows. Logical PK = (run_id, product_id, bar_date, dq_reason).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_slv_crypto_ohlc_quarantine_keys": """
            ALTER TABLE `{catalog}`.slv_market.crypto_ohlc_1d_quarantine
            ADD CONSTRAINT ck_slv_crypto_ohlc_quarantine_keys
            CHECK (
                run_id IS NOT NULL AND product_id IS NOT NULL AND bar_date IS NOT NULL AND
                dq_reason IS NOT NULL AND source_ingested_at IS NOT NULL AND
                source_run_id IS NOT NULL AND quarantined_at IS NOT NULL
            )
            """,
        },
    },
    "gld_market.dp_crypto_returns_1d": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_market.dp_crypto_returns_1d (
            product_id STRING NOT NULL COMMENT 'Logical PK part 1',
            bar_date DATE NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING NOT NULL,
            quote_currency STRING NOT NULL,
            close DOUBLE,
            simple_return_1d DOUBLE,
            log_return_1d DOUBLE,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold market product: daily crypto returns. Logical PK = (product_id, bar_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_gld_crypto_returns_keys": """
            ALTER TABLE `{catalog}`.gld_market.dp_crypto_returns_1d
            ADD CONSTRAINT ck_gld_crypto_returns_keys
            CHECK (
                product_id IS NOT NULL AND bar_date IS NOT NULL AND
                base_asset IS NOT NULL AND quote_currency IS NOT NULL
            )
            """,
            "ck_gld_crypto_returns_close": """
            ALTER TABLE `{catalog}`.gld_market.dp_crypto_returns_1d
            ADD CONSTRAINT ck_gld_crypto_returns_close
            CHECK (close IS NULL OR close >= 0)
            """,
        },
    },
    "gld_market.dp_crypto_volatility_1d": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_market.dp_crypto_volatility_1d (
            product_id STRING NOT NULL COMMENT 'Logical PK part 1',
            bar_date DATE NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING NOT NULL,
            quote_currency STRING NOT NULL,
            simple_return_1d DOUBLE,
            volatility_7d DOUBLE,
            volatility_30d DOUBLE,
            volatility_90d DOUBLE,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold market product: daily realized volatility. Logical PK = (product_id, bar_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_gld_crypto_volatility_keys": """
            ALTER TABLE `{catalog}`.gld_market.dp_crypto_volatility_1d
            ADD CONSTRAINT ck_gld_crypto_volatility_keys
            CHECK (
                product_id IS NOT NULL AND bar_date IS NOT NULL AND
                base_asset IS NOT NULL AND quote_currency IS NOT NULL
            )
            """,
            "ck_gld_crypto_volatility_values": """
            ALTER TABLE `{catalog}`.gld_market.dp_crypto_volatility_1d
            ADD CONSTRAINT ck_gld_crypto_volatility_values
            CHECK (
                (volatility_7d IS NULL OR volatility_7d >= 0) AND
                (volatility_30d IS NULL OR volatility_30d >= 0) AND
                (volatility_90d IS NULL OR volatility_90d >= 0)
            )
            """,
        },
    },
    "brz_macro.raw_ecb_fx_ref_rates_daily": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_macro.raw_ecb_fx_ref_rates_daily (
            base_currency STRING NOT NULL COMMENT 'Typically EUR',
            quote_currency STRING NOT NULL COMMENT 'Quoted currency, e.g., USD',
            rate_date DATE NOT NULL COMMENT 'FX reference rate date',
            rate DOUBLE COMMENT 'ECB reference FX rate',
            ingested_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp (UTC)',
            run_id STRING NOT NULL COMMENT 'Pipeline run identifier',
            payload_hash STRING COMMENT 'Hash of raw source payload for audit/dedup'
        )
        USING DELTA
        COMMENT 'Bronze: raw ECB FX reference rates (daily). Logical PK = (base_currency, quote_currency, rate_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_ecb_fx_keys": """
            ALTER TABLE `{catalog}`.brz_macro.raw_ecb_fx_ref_rates_daily
            ADD CONSTRAINT ck_ecb_fx_keys
            CHECK (base_currency IS NOT NULL AND quote_currency IS NOT NULL AND rate_date IS NOT NULL)
            """,
            "ck_ecb_fx_rate": """
            ALTER TABLE `{catalog}`.brz_macro.raw_ecb_fx_ref_rates_daily
            ADD CONSTRAINT ck_ecb_fx_rate
            CHECK (rate IS NULL OR rate > 0)
            """,
        },
    },
    "slv_macro.ecb_fx_ref_rates_daily": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_macro.ecb_fx_ref_rates_daily (
            base_currency STRING NOT NULL COMMENT 'Logical PK part 1',
            quote_currency STRING NOT NULL COMMENT 'Logical PK part 2',
            rate_date DATE NOT NULL COMMENT 'Logical PK part 3',
            rate DOUBLE,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Silver: cleaned ECB FX daily rates. Logical PK = (base_currency, quote_currency, rate_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_slv_ecb_fx_keys": """
            ALTER TABLE `{catalog}`.slv_macro.ecb_fx_ref_rates_daily
            ADD CONSTRAINT ck_slv_ecb_fx_keys
            CHECK (base_currency IS NOT NULL AND quote_currency IS NOT NULL AND rate_date IS NOT NULL)
            """,
            "ck_slv_ecb_fx_rate": """
            ALTER TABLE `{catalog}`.slv_macro.ecb_fx_ref_rates_daily
            ADD CONSTRAINT ck_slv_ecb_fx_rate
            CHECK (rate IS NULL OR rate > 0)
            """,
        },
    },
    "brz_macro.raw_fred_series": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_macro.raw_fred_series (
            series_id STRING NOT NULL COMMENT 'FRED series id, e.g., CPIAUCSL',
            observation_date DATE NOT NULL COMMENT 'Observation date',
            realtime_start DATE NOT NULL COMMENT 'Revision window start (FRED realtime)',
            realtime_end DATE NOT NULL COMMENT 'Revision window end (FRED realtime)',
            value DOUBLE COMMENT 'Numeric observation value',
            ingested_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp (UTC)',
            run_id STRING NOT NULL COMMENT 'Pipeline run identifier',
            payload_hash STRING COMMENT 'Hash of raw payload for audit/dedup'
        )
        USING DELTA
        COMMENT 'Bronze: raw FRED observations (long table). Logical PK = (series_id, observation_date, realtime_start, realtime_end).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_fred_keys": """
            ALTER TABLE `{catalog}`.brz_macro.raw_fred_series
            ADD CONSTRAINT ck_fred_keys
            CHECK (
                series_id IS NOT NULL AND observation_date IS NOT NULL AND
                realtime_start IS NOT NULL AND realtime_end IS NOT NULL
            )
            """,
            "ck_fred_realtime_order": """
            ALTER TABLE `{catalog}`.brz_macro.raw_fred_series
            ADD CONSTRAINT ck_fred_realtime_order
            CHECK (realtime_start <= realtime_end)
            """,
        },
    },
    "slv_macro.fred_series_clean": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_macro.fred_series_clean (
            series_id STRING NOT NULL COMMENT 'Logical PK part 1',
            observation_date DATE NOT NULL COMMENT 'Logical PK part 2',
            realtime_start DATE NOT NULL COMMENT 'Logical PK part 3',
            realtime_end DATE NOT NULL COMMENT 'Logical PK part 4',
            value DOUBLE,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Silver: cleaned FRED series without resampling. Logical PK = (series_id, observation_date, realtime_start, realtime_end).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_slv_fred_keys": """
            ALTER TABLE `{catalog}`.slv_macro.fred_series_clean
            ADD CONSTRAINT ck_slv_fred_keys
            CHECK (
                series_id IS NOT NULL AND observation_date IS NOT NULL AND
                realtime_start IS NOT NULL AND realtime_end IS NOT NULL
            )
            """,
            "ck_slv_fred_realtime_order": """
            ALTER TABLE `{catalog}`.slv_macro.fred_series_clean
            ADD CONSTRAINT ck_slv_fred_realtime_order
            CHECK (realtime_start <= realtime_end)
            """,
        },
    },
    "gld_macro.dp_macro_indicators": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_macro.dp_macro_indicators (
            indicator_id STRING NOT NULL COMMENT 'Logical PK part 1',
            observation_date DATE NOT NULL COMMENT 'Logical PK part 2',
            source_system STRING NOT NULL COMMENT 'External source namespace like ecb or fred',
            indicator_group STRING COMMENT 'Logical family like fx_ref_rate, inflation, or policy_rate',
            value DOUBLE COMMENT 'Numeric indicator value in the stated unit',
            unit STRING NOT NULL COMMENT 'Display unit like USD per EUR or Index 1982-1984=100',
            frequency STRING COMMENT 'Source frequency like D, W, M, or Q',
            is_official BOOLEAN NOT NULL COMMENT 'True when the row comes directly from the external source',
            derivation_method STRING NOT NULL COMMENT 'official_source, inverse_rate, spread, or other internal derivation',
            derived_from_indicator_id STRING COMMENT 'Upstream indicator id when the row is internally derived',
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold macro product stored as a long table with source and derivation provenance. Logical PK = (indicator_id, observation_date).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_gld_macro_indicator_keys": """
            ALTER TABLE `{catalog}`.gld_macro.dp_macro_indicators
            ADD CONSTRAINT ck_gld_macro_indicator_keys
            CHECK (
                indicator_id IS NOT NULL AND observation_date IS NOT NULL AND source_system IS NOT NULL AND
                unit IS NOT NULL AND is_official IS NOT NULL AND derivation_method IS NOT NULL
            )
            """,
            "ck_gld_macro_indicator_provenance": """
            ALTER TABLE `{catalog}`.gld_macro.dp_macro_indicators
            ADD CONSTRAINT ck_gld_macro_indicator_provenance
            CHECK ((is_official = true AND derived_from_indicator_id IS NULL) OR is_official = false)
            """,
        },
    },
    "gld_cross.dp_crypto_macro_features_1d": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_cross.dp_crypto_macro_features_1d (
            feature_date DATE NOT NULL COMMENT 'Logical PK part 1',
            product_id STRING NOT NULL COMMENT 'Logical PK part 2',
            base_asset STRING,
            quote_currency STRING,
            simple_return_1d DOUBLE,
            log_return_1d DOUBLE,
            volatility_7d DOUBLE,
            volatility_30d DOUBLE,
            macro_features MAP<STRING, DOUBLE>,
            fill_policy STRING,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        )
        USING DELTA
        COMMENT 'Gold cross-domain feature table. Logical PK = (feature_date, product_id).'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_gld_cross_keys": """
            ALTER TABLE `{catalog}`.gld_cross.dp_crypto_macro_features_1d
            ADD CONSTRAINT ck_gld_cross_keys
            CHECK (feature_date IS NOT NULL AND product_id IS NOT NULL)
            """,
            "ck_gld_cross_volatility": """
            ALTER TABLE `{catalog}`.gld_cross.dp_crypto_macro_features_1d
            ADD CONSTRAINT ck_gld_cross_volatility
            CHECK (
                (volatility_7d IS NULL OR volatility_7d >= 0) AND
                (volatility_30d IS NULL OR volatility_30d >= 0)
            )
            """,
        },
    },
    "obs.obs_ingestion_state": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.obs.obs_ingestion_state (
            pipeline_name STRING NOT NULL,
            source_name STRING,
            target_table STRING NOT NULL,
            watermark_value STRING,
            watermark_type STRING,
            last_success_at TIMESTAMP,
            last_run_id STRING,
            status STRING,
            updated_at TIMESTAMP NOT NULL
        )
        USING DELTA
        COMMENT 'Latest ingestion checkpoint state per pipeline.'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_obs_ingestion_state_keys": """
            ALTER TABLE `{catalog}`.obs.obs_ingestion_state
            ADD CONSTRAINT ck_obs_ingestion_state_keys
            CHECK (pipeline_name IS NOT NULL AND target_table IS NOT NULL AND updated_at IS NOT NULL)
            """,
        },
    },
    "obs.obs_pipeline_run_log": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.obs.obs_pipeline_run_log (
            run_id STRING NOT NULL,
            pipeline_name STRING NOT NULL,
            layer STRING NOT NULL,
            source_name STRING,
            target_table STRING NOT NULL,
            status STRING NOT NULL,
            rows_read BIGINT,
            rows_written BIGINT,
            started_at TIMESTAMP NOT NULL,
            completed_at TIMESTAMP,
            error_message STRING,
            metadata_json STRING
        )
        USING DELTA
        COMMENT 'Pipeline execution log for observability and troubleshooting.'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        """,
        "constraints": {
            "ck_obs_pipeline_run_log_keys": """
            ALTER TABLE `{catalog}`.obs.obs_pipeline_run_log
            ADD CONSTRAINT ck_obs_pipeline_run_log_keys
            CHECK (
                run_id IS NOT NULL AND pipeline_name IS NOT NULL AND layer IS NOT NULL AND
                target_table IS NOT NULL AND status IS NOT NULL AND started_at IS NOT NULL
            )
            """,
            "ck_obs_pipeline_run_log_rows": """
            ALTER TABLE `{catalog}`.obs.obs_pipeline_run_log
            ADD CONSTRAINT ck_obs_pipeline_run_log_rows
            CHECK (
                (rows_read IS NULL OR rows_read >= 0) AND
                (rows_written IS NULL OR rows_written >= 0)
            )
            """,
        },
    },
}

for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{schema}")

tables_ensured = []
constraints_added = []
constraints_existing = []

for table_name, spec in table_specs.items():
    full_name = f"{catalog}.{table_name}"
    spark.sql(spec["ddl"].format(catalog=catalog))
    tables_ensured.append(full_name)

    for constraint_name, constraint_sql in spec["constraints"].items():
        if constraint_exists(full_name, constraint_name):
            constraints_existing.append(f"{full_name}:{constraint_name}")
            continue

        spark.sql(constraint_sql.format(catalog=catalog))
        constraints_added.append(f"{full_name}:{constraint_name}")

result = {
    "status": "success",
    "catalog": catalog,
    "schemas_created": [f"{catalog}.{schema}" for schema in schemas],
    "tables_ensured": tables_ensured,
    "constraints_added": constraints_added,
    "constraints_existing": constraints_existing,
    "table_count": len(table_specs),
}

result